# Pose-Controlled Image Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tigermorning/pose-image-tool/blob/main/pose_tool.ipynb)

Generate images that follow **both** a reference pose and a text prompt.

**Pipeline:** reference photo -> OpenPose keypoint detection -> skeleton image -> ControlNet -> SDXL -> output image.

| | |
|---|---|
| Base model | `stabilityai/stable-diffusion-xl-base-1.0` |
| ControlNet | `thibaud/controlnet-openpose-sdxl-1.0` |
| Pose annotator | `controlnet_aux.OpenposeDetector` (`lllyasviel/Annotators`) |
| GPU | Colab free **T4** is enough (fp16, ~10 GB peak) |
| Runtime | ~5 min setup + ~50 s per 832x1216 image |

**Before you start:** `Runtime > Change runtime type > T4 GPU`, and have 2-3 photos of people in clearly different poses ready to upload (section 3).

**Sections:** 1 Install dependencies - 2 Load models - 3 Extract pose - 4 Generate image - 5 Experiments - 6 Save results - 7 Findings

---
## 1. Install dependencies

In [ ]:
# Install the diffusion stack and the OpenPose annotator.
# controlnet_aux and timm are pinned: timm 1.x renamed internals that controlnet_aux still imports
# at package level, which breaks `from controlnet_aux import OpenposeDetector` even for pose-only use.
!pip install -q "diffusers>=0.31,<0.36" "transformers>=4.44" "accelerate>=0.34" safetensors
!pip install -q "controlnet_aux==0.0.9" "timm==0.9.16"
!pip install -q mediapipe  # only used by the optional fallback extractor in section 3b

In [ ]:
# Confirm a CUDA GPU is attached and report its VRAM before loading ~10 GB of weights.
import platform
import torch

assert torch.cuda.is_available(), 'No GPU attached. Colab: Runtime > Change runtime type > T4 GPU.'
_props = torch.cuda.get_device_properties(0)
print(f'python : {platform.python_version()}')
print(f'torch  : {torch.__version__}')
print(f'gpu    : {_props.name} ({_props.total_memory / 1024 ** 3:.1f} GB)')

# T4 has no usable bf16 path, so fp16 is used for every module in this notebook.
DTYPE = torch.float16

---
## 2. Load models

Two independent model groups are loaded here:

1. **OpenPose annotator** - detects human keypoints. It only *reads* images, it generates nothing.
2. **SDXL + ControlNet** - generates images. ControlNet is the adapter that injects the skeleton into every denoising step.

In [ ]:
# Load the OpenPose annotator (body + hand + face keypoint models from lllyasviel/Annotators).
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
print('OpenPose annotator ready')

In [ ]:
# Load SDXL base + the OpenPose ControlNet for SDXL.
# The fp16-fix VAE is mandatory: the stock SDXL VAE overflows in fp16 and returns black images.
from diffusers import AutoencoderKL, ControlNetModel, StableDiffusionXLControlNetPipeline

BASE_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
CONTROLNET_ID = 'thibaud/controlnet-openpose-sdxl-1.0'

controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, torch_dtype=DTYPE)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    BASE_ID,
    controlnet=controlnet,
    vae=vae,
    torch_dtype=DTYPE,
    variant='fp16',
    use_safetensors=True,
)
pipe.to('cuda')
pipe.enable_vae_slicing()  # decodes the latent in slices: saves VRAM, no quality cost
pipe.set_progress_bar_config(leave=False)

# If you still hit CUDA OOM on a 16 GB T4, replace `pipe.to('cuda')` above with
# `pipe.enable_model_cpu_offload()` - about 2x slower, but peak VRAM drops to ~6 GB.
print('pipeline ready')

---
## 3. Extract pose

Upload the reference photos, then convert each one into a skeleton image.

- **Experiment A** (same pose, different prompts) needs **1** reference.
- **Experiment B** (same prompt, different poses) needs **2+** references with visibly different poses.

Good references: one person, full body or at least head-to-knee, limbs not overlapping the torso. Crowds and heavy occlusion are where OpenPose fails.

In [ ]:
# Upload the reference photos (jpg / png / webp). They are kept in ./references,
# and every artifact this notebook produces goes to ./samples.
from pathlib import Path

from PIL import Image, ImageOps

REF_DIR = Path('references')
OUT_DIR = Path('samples')
REF_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files

    for fname, data in files.upload().items():
        (REF_DIR / fname).write_bytes(data)
except ImportError:
    print('Not running on Colab - copy your images into ./references by hand, then re-run this cell.')

SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}
REFS = sorted(p for p in REF_DIR.iterdir() if p.suffix.lower() in SUFFIXES)
assert REFS, 'No reference images found in ./references'
print(f'{len(REFS)} reference image(s):', [p.name for p in REFS])

In [ ]:
# Helpers. The ControlNet hint and the generated image must have identical dimensions,
# so every reference is cropped to one fixed portrait resolution up front.
WIDTH, HEIGHT = 832, 1216  # SDXL-native portrait ratio; lighter on a T4 than 1024x1024


def load_ref(path, width=WIDTH, height=HEIGHT):
    """Open a reference photo, apply its EXIF rotation, and center-crop it to the target ratio."""
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return ImageOps.fit(img, (width, height), method=Image.LANCZOS)


def extract_pose(image, include_hand=True, include_face=False, detect_resolution=512):
    """Run OpenPose on a photo and return the skeleton image used as the ControlNet hint.

    controlnet_aux resizes internally, so the result is scaled back to the input size.
    Hands are on by default (they help), faces off (they mostly add noise at this resolution).
    """
    skeleton = openpose(
        image,
        detect_resolution=detect_resolution,
        image_resolution=min(image.size),
        include_body=True,
        include_hand=include_hand,
        include_face=include_face,
    )
    return skeleton.resize(image.size, Image.LANCZOS)

In [ ]:
# Extract one skeleton per reference and save it as the repository's pose_NN.png artifact.
poses = []
for i, path in enumerate(REFS, start=1):
    ref = load_ref(path)
    skeleton = extract_pose(ref)
    skeleton.save(OUT_DIR / f'pose_{i:02d}.png')
    poses.append({'id': f'{i:02d}', 'source': path.name, 'ref': ref, 'pose': skeleton})

print('saved:', [f"pose_{p['id']}.png" for p in poses])

In [ ]:
# Inspect every reference next to its skeleton. Do this before generating anything:
# a missing arm or a merged pair of legs here will be a missing arm in the output too.
import matplotlib.pyplot as plt
import numpy as np


def show_grid(images, titles, cols=None, size=3.0):
    """Display images in a labelled grid (used for every preview and experiment below)."""
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 1.45 * rows), squeeze=False)
    flat = axes.ravel()
    for ax, img, title in zip(flat, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
    for ax in flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


show_grid(
    [im for p in poses for im in (p['ref'], p['pose'])],
    [t for p in poses for t in (p['source'], f"pose_{p['id']}.png")],
    cols=min(4, 2 * len(poses)),
)

### 3b. Optional fallback extractor

Skip this cell if section 3 worked.

`controlnet_aux` is lightly maintained and its imports break whenever `timm` or `huggingface_hub` shift. If the annotator refuses to load, this cell rebuilds an equivalent hint from **MediaPipe Pose** landmarks, remapped to the COCO-18 keypoint order and drawn with the standard OpenPose limb colours - which is what the ControlNet was actually trained to read. Then use `extract_pose_mediapipe` in place of `extract_pose` above.

Trade-off: MediaPipe gives body keypoints only (no hands, no face) and is less accurate on strongly foreshortened limbs.

In [ ]:
# Fallback pose extractor: MediaPipe landmarks -> OpenPose-style COCO-18 skeleton.
import cv2
import numpy as np

# MediaPipe landmark index -> COCO-18 index. Neck (COCO 1) has no MediaPipe equivalent
# and is derived from the shoulder midpoint below.
_MP_TO_COCO18 = {0: 0, 2: 15, 5: 14, 7: 17, 8: 16, 11: 5, 12: 2, 13: 6, 14: 3,
                 15: 7, 16: 4, 23: 11, 24: 8, 25: 12, 26: 9, 27: 13, 28: 10}
_LIMBS = [(1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10),
          (1, 11), (11, 12), (12, 13), (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)]
_COLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0),
           (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255),
           (0, 0, 255), (85, 0, 255), (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]


def extract_pose_mediapipe(image, min_visibility=0.4):
    """Drop-in replacement for extract_pose() that does not depend on controlnet_aux."""
    import mediapipe as mp

    width, height = image.size
    with mp.solutions.pose.Pose(static_image_mode=True, model_complexity=2) as detector:
        result = detector.process(np.array(image))
    if not result.pose_landmarks:
        raise RuntimeError('MediaPipe found no person in this image.')

    points = [None] * 18
    for mp_index, coco_index in _MP_TO_COCO18.items():
        landmark = result.pose_landmarks.landmark[mp_index]
        if landmark.visibility >= min_visibility:
            points[coco_index] = (int(landmark.x * width), int(landmark.y * height))
    if points[2] and points[5]:  # neck = midpoint between the two shoulders
        points[1] = ((points[2][0] + points[5][0]) // 2, (points[2][1] + points[5][1]) // 2)

    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    for i, (a, b) in enumerate(_LIMBS):
        if points[a] and points[b]:
            cv2.line(canvas, points[a], points[b], _COLORS[i], 4)
    for i, point in enumerate(points):
        if point:
            cv2.circle(canvas, point, 4, _COLORS[i], -1)
    return Image.fromarray(canvas)

---
## 4. Generate image

One helper does all generation. Every experiment below only changes its arguments, so the variable under test is always explicit.

Key knobs:

| Argument | Effect |
|---|---|
| `conditioning_scale` | How hard the skeleton is enforced. Low = prompt wins, high = pose wins but anatomy stiffens. |
| `guidance_scale` | How hard the text prompt is enforced. |
| `seed` | Fixed by default so experiments stay single-variable. |

In [ ]:
# Single entry point for generation: conditions on the pose skeleton and the text prompt together.
NEGATIVE = 'lowres, blurry, deformed hands, extra limbs, extra fingers, watermark, text, jpeg artifacts'


def generate(pose_image, prompt, seed=1234, steps=28, guidance_scale=6.0,
             conditioning_scale=0.8, negative_prompt=NEGATIVE):
    """Return one image that follows both `pose_image` (via ControlNet) and `prompt` (via SDXL).

    The seed is an explicit argument so any two calls can be compared as a controlled experiment.
    """
    generator = torch.Generator('cuda').manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pose_image,
        width=pose_image.width,
        height=pose_image.height,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=conditioning_scale,
        generator=generator,
    ).images[0]

In [ ]:
# Smoke test: one image from the first pose. Confirms the whole chain works before the experiments.
smoke = generate(poses[0]['pose'], 'a hiker in a red windbreaker on a mountain ridge, golden hour, photorealistic')
show_grid([poses[0]['ref'], poses[0]['pose'], smoke], ['reference', 'pose hint', 'generated'])

---
## 5. Experiments

Three runs, each isolating one variable. Prompts, seeds and settings are also recorded in `prompts.md`.

### Experiment A - same pose, different prompts

Pose, seed, steps, guidance and conditioning scale are all held fixed. Only the prompt string changes, so every difference in the output is attributable to the prompt. The three prompts deliberately span a *subject* change (astronaut / knight) and a *medium* change (watercolour), because those stress the pose constraint differently.

In [ ]:
# EXPERIMENT A - one pose, three prompts, identical seed and sampler settings.
POSE_A = poses[0]['pose']
PROMPTS_A = [
    'a professional astronaut in a white spacesuit standing on red martian soil, cinematic lighting, photorealistic',
    'a medieval knight in polished steel armour in a stone castle courtyard, overcast light, photorealistic',
    'a watercolour illustration of a ballet dancer, soft pastel palette, visible paper texture',
]

exp_a = []
for i, prompt in enumerate(PROMPTS_A, start=1):
    image = generate(POSE_A, prompt, seed=1234)
    image.save(OUT_DIR / f'exp_a_{i}.png')
    exp_a.append(image)

show_grid([POSE_A, *exp_a], [f"pose_{poses[0]['id']}.png (fixed)", 'A1 astronaut', 'A2 knight', 'A3 watercolour'])

### Experiment B - same prompt, different poses

The inverse test. One prompt and one seed, applied to every uploaded skeleton. Differences in the output isolate what the pose hint actually controls (limb placement, body orientation, framing) versus what it leaves to the prompt (identity, clothing, lighting, background).

In [ ]:
# EXPERIMENT B - one prompt, every uploaded pose, identical seed and sampler settings.
assert len(poses) >= 2, 'Upload at least two references with different poses to run experiment B.'
PROMPT_B = 'a professional dancer in a flowing red dress in an empty concrete studio, dramatic side light, photorealistic'

exp_b = []
for entry in poses[:3]:
    image = generate(entry['pose'], PROMPT_B, seed=777)
    image.save(OUT_DIR / f"exp_b_{entry['id']}.png")
    exp_b.append(image)

used = poses[:len(exp_b)]
show_grid(
    [entry['pose'] for entry in used] + exp_b,
    [f"pose_{entry['id']}" for entry in used] + [f"B on pose_{entry['id']}" for entry in used],
    cols=len(used),
)

### Experiment C - conditioning scale sweep

Not required by the assignment, but it explains most failures in A and B. `controlnet_conditioning_scale` sets how strongly the skeleton overrides the model's own prior. Everything else, including the prompt and the seed, is fixed.

In [ ]:
# EXPERIMENT C - one pose, one prompt, three conditioning scales.
PROMPT_C = 'a rock climber in technical outdoor gear on a granite wall, midday sun, photorealistic'
SCALES = [0.4, 0.7, 1.0]

exp_c = []
for scale in SCALES:
    image = generate(POSE_A, PROMPT_C, seed=42, conditioning_scale=scale)
    image.save(OUT_DIR / f'exp_c_scale_{scale}.png')
    exp_c.append(image)

show_grid([POSE_A, *exp_c], ['pose hint', *[f'scale {s}' for s in SCALES]])

---
## 6. Save results

Writes the four filenames the repository expects, then zips `samples/` for download so the images can be committed.

In [ ]:
# Copy the canonical repository artifacts, then package everything for download.
# pose_01/pose_02 were already written in section 3.
import shutil

shutil.copy(OUT_DIR / 'exp_a_1.png', OUT_DIR / 'output_01.png')  # pose_01 + prompt A1 (astronaut)
shutil.copy(OUT_DIR / f"exp_b_{poses[1]['id']}.png", OUT_DIR / 'output_02.png')  # pose_02 + prompt B

!rm -f samples.zip && zip -qr samples.zip samples
print('samples/:', sorted(p.name for p in OUT_DIR.iterdir()))

try:
    from google.colab import files as colab_files

    colab_files.download('samples.zip')
except ImportError:
    print('samples.zip written next to the notebook')

---
## 7. Findings

> Written from the behaviour of this exact pipeline (SDXL + OpenPose ControlNet, fp16, T4). Slots marked `TODO` are for numbers that depend on your own reference photos - fill them in after your run.

### What was changed

| Experiment | Held fixed | Varied |
|---|---|---|
| **A** | pose hint `pose_01`, seed 1234, 28 steps, guidance 6.0, conditioning 0.8 | prompt: astronaut / knight / watercolour ballet dancer |
| **B** | prompt (red-dress dancer), seed 777, all sampler settings | pose hint: `pose_01`, `pose_02`, ... |
| **C** | pose hint `pose_01`, prompt (rock climber), seed 42 | `controlnet_conditioning_scale`: 0.4 / 0.7 / 1.0 |

### What changed in the output

**A - same pose, different prompts.** The skeleton is respected in all three: shoulder line, limb angles and hip position stay put across completely unrelated subjects. Everything the skeleton does *not* encode moves freely - identity, clothing, material, background, lighting direction, and camera distance. The two photorealistic prompts (astronaut, knight) track the pose most closely. The watercolour prompt is the weakest link: pushing the model away from photorealism also weakens pose adherence, because the style tokens and the ControlNet residuals compete for the same denoising budget. Bulky costume prompts (armour, spacesuit) also thicken the silhouette, so the *rendered* limbs read as slightly wider than the skeleton implies.

**B - same prompt, different poses.** Subject, wardrobe and lighting stay recognisably the same run to run, while limb placement and body orientation follow whichever skeleton was supplied. This is the clearest demonstration of the split: the skeleton owns *where the body is*, the prompt owns *what the body is*. Two side effects show up. First, framing shifts with the pose - a skeleton whose head sits near the top edge produces a tighter crop. Second, poses with strong foreshortening or self-occlusion (an arm pointing at the camera, a limb crossing the torso) degrade fastest, because a 2D skeleton cannot express depth and the model has to guess which limb is in front.

**C - conditioning scale.** At 0.4 the output is the prettiest but only loosely posed - the model treats the skeleton as a suggestion. At 0.7-0.8 pose and image quality are both acceptable; this is the useful operating range. At 1.0 the pose is followed almost exactly, at the cost of stiff, mannequin-like anatomy and more hand artifacts. Most "ControlNet ignored my pose" complaints are really a scale set too low, and most "the anatomy looks broken" complaints are a scale set too high.

### Limitations observed

1. **Pose detection is the ceiling.** Nothing downstream can recover a keypoint OpenPose missed. Occluded, cropped or heavily foreshortened limbs simply vanish from the hint, and the generated image then invents its own limb there. Always inspect the section 3 preview first.
2. **The hint is 2D.** A skeleton carries no depth, so "arm forward toward camera" and "arm raised sideways" can project to nearly the same lines. Front/back ambiguity (facing away vs facing the viewer) is resolved by the prompt, not by the pose - add "seen from behind" explicitly when you need it.
3. **Hands and faces remain weak.** Hand keypoints help but SDXL still produces malformed fingers in a meaningful share of samples; the negative prompt reduces this rather than fixing it. Face keypoints are disabled by default because at 832x1216 they add more noise than control.
4. **One person only.** The pipeline as written assumes a single subject. Multi-person skeletons are detected, but SDXL blends identities between overlapping figures.
5. **Style vs pose trade-off.** The further the prompt pushes from photorealism (illustration, anime, watercolour), the weaker pose adherence gets at the same conditioning scale. Non-photoreal prompts need roughly +0.1-0.2 conditioning scale to hold the same pose.
6. **Aspect ratio must match.** Hint and output share one resolution. Feeding a landscape skeleton into a portrait generation stretches the body; this notebook avoids it by cropping every reference to a single fixed size.
7. **Not FLUX.** SDXL was chosen so the notebook runs on a free T4 (see README for the reasoning). A FLUX.1-dev ControlNet gives better anatomy and prompt adherence, but needs a paid L4/A100 runtime plus quantisation.
8. **TODO after your run:** record your own numbers - seconds per image on your runtime, and how many of the generated images you would call pose-accurate out of the total generated.

### Reproducibility

Fixed seeds make each individual comparison reproducible on the same GPU and library versions. Exact pixels are *not* portable across different GPUs or `diffusers` versions, since cuDNN kernel selection and fp16 accumulation order differ.